<a href="https://colab.research.google.com/github/datasentient1/AutoGPT-MessingAround/blob/master/colab_mvp_pipeline_v0.1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Verifier-Guided Reasoning MVP

This notebook is the public Colab orchestration surface for the project. It keeps the repository's legacy name, but the actual implementation is positioned as a verifier-guided process supervision system rather than a full RLHF stack.

Notebook goals:

- install the repo and Colab extras,
- mount Google Drive,
- configure DVC and MLflow paths on Drive,
- run the arithmetic verifier demo,
- preview the SFT path for `Qwen/Qwen2.5-1.5B-Instruct`.


In [19]:
import os

# 1. Generate a new SSH key (if it doesn't exist)
ssh_path = os.path.expanduser('~/.ssh/id_ed25519')
if not os.path.exists(ssh_path):
    !ssh-keygen -t ed25519 -C "colab-session" -N "" -f {ssh_path}

# 2. Add github.com to known_hosts to avoid the interactive prompt
!ssh-keyscan -t ed25519 github.com >> ~/.ssh/known_hosts

# 3. Print the Public Key
print("\n--- COPY THE KEY BELOW ---\n")
with open(ssh_path + '.pub', 'r') as f:
    print(f.read())
print("--- END OF KEY ---\n")
print("Go to https://github.com/settings/ssh/new and paste the key above.")

# github.com:22 SSH-2.0-a59182e

--- COPY THE KEY BELOW ---

ssh-ed25519 KEY_VALUE_HERE colab-session

--- END OF KEY ---

Go to https://github.com/settings/ssh/new and paste the key above.


In [20]:
# 4. Test the connection
!ssh -T git@github.com

Hi datasentient1! You've successfully authenticated, but GitHub does not provide shell access.


In [22]:
# Import userdata for secure access to secrets
from google.colab import userdata

# Retrieve the GitHub Personal Access Token from Colab secrets
GH_PAT = userdata.get('GH_PAT')

# Re-execute cell mbgtTcN_PhAS to use the PAT for cloning
# This cell is designed to be temporary, and its content will be integrated into mbgtTcN_PhAS.
# The actual modification will happen in the next step to clean up the notebook.
# This is a placeholder to show the user how the PAT would be used.
print("Please ensure you've saved your GitHub PAT as 'GH_PAT' in Colab Secrets.")
print("Once confirmed, I will modify cell mbgtTcN_PhAS to use it.")

Please ensure you've saved your GitHub PAT as 'GH_PAT' in Colab Secrets.
Once confirmed, I will modify cell mbgtTcN_PhAS to use it.


### SSH Authentication Setup

After successful authentication above, we can update the clone command to use the SSH URL.

In [23]:
import os
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
print({'in_colab': IN_COLAB, 'python': sys.version.split()[0]})

if IN_COLAB:
    # The folder created by cloning this specific repo
    repo_name = 'RLHF-Logic-Verification-Framework'
    repo_url = 'git@github.com:datasentient1/RLHF-Logic-Verification-Framework.git'

    # Clone the repository if it doesn't exist
    if not Path(repo_name).exists():
        print(f"Cloning {repo_url} via SSH...")
        !git clone {repo_url}

    # Change directory into the cloned repository
    if Path(repo_name).exists():
        os.chdir(repo_name)
        print(f"Changed directory to {os.getcwd()}")
    else:
        print(f"Error: Directory {repo_name} not found. Please ensure the SSH key is added to GitHub and the connection test passed.")
        sys.exit(1)

    # Now run the pip install command from within the project directory
    get_ipython().run_line_magic('pip', 'install -q -e .[colab,dev]')


{'in_colab': True, 'python': '3.12.13'}
Cloning git@github.com:datasentient1/RLHF-Logic-Verification-Framework.git via SSH...
Cloning into 'RLHF-Logic-Verification-Framework'...
remote: Enumerating objects: 75, done.
remote: Counting objects: 100% (75/75), done.
remote: Compressing objects: 100% (67/67), done.
remote: Total 75 (delta 10), reused 69 (delta 8), pack-reused 0 (from 0)
Receiving objects: 100% (75/75), 1.45 MiB | 15.97 MiB/s, done.
Resolving deltas: 100% (10/10), done.
Changed directory to /content/RLHF-Logic-Verification-Framework/RLHF-Logic-Verification-Framework
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for verifier-guided-reasoning (pyproject.toml) ... done


In [25]:
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

DVC_REMOTE_PATH = Path('/content/drive/MyDrive/rlhf_logic_verification/dvc')
MLFLOW_ROOT = Path('/content/drive/MyDrive/rlhf_logic_verification/mlruns')
DVC_REMOTE_PATH.mkdir(parents=True, exist_ok=True)
MLFLOW_ROOT.mkdir(parents=True, exist_ok=True)
print({'dvc_remote': str(DVC_REMOTE_PATH), 'mlflow_root': str(MLFLOW_ROOT)})


Mounted at /content/drive
{'dvc_remote': '/content/drive/MyDrive/rlhf_logic_verification/dvc', 'mlflow_root': '/content/drive/MyDrive/rlhf_logic_verification/mlruns'}


In [26]:
if IN_COLAB:
    get_ipython().system('dvc init || true')
    get_ipython().system(f'dvc remote add -d colab_drive {DVC_REMOTE_PATH} || true')

os.environ['MLFLOW_TRACKING_URI'] = MLFLOW_ROOT.resolve().as_uri()
print('MLFLOW_TRACKING_URI=', os.environ['MLFLOW_TRACKING_URI'])


Initialized DVC repository.

You can now commit the changes to git.

+---------------------------------------------------------------------+
|                                                                     |
|        DVC has enabled anonymous aggregate usage analytics.         |
|     Read the analytics documentation (and how to opt-out) here:     |
|             <https://dvc.org/doc/user-guide/analytics>              |
|                                                                     |
+---------------------------------------------------------------------+

What's next?
------------
- Check out the documentation: <https://dvc.org/doc>
- Get help and share ideas: <https://dvc.org/chat>
- Star us on GitHub: <https://github.com/treeverse/dvc>
Setting 'colab_drive' as a default remote.
MLFLOW_TRACKING_URI= file:///content/drive/MyDrive/rlhf_logic_verification/mlruns


In [28]:
import sys
import os
from pathlib import Path

# Ensure the src directory is in the path so the package can be found
repo_root = Path('/content/RLHF-Logic-Verification-Framework')
if repo_root.exists():
    src_path = str(repo_root / 'src')
    if src_path not in sys.path:
        sys.path.append(src_path)

from verifier_guided_reasoning.pipeline import run_small_demo

summary = run_small_demo(
    output_path='artifacts/eval/demo_summary.json',
    report_markdown_path='artifacts/eval/demo_report.md',
    tracker_root=str(MLFLOW_ROOT) if 'MLFLOW_ROOT' in locals() else 'mlruns',
)
summary

/usr/local/lib/python3.12/dist-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
2026/04/26 01:12:26 INFO mlflow.tracking.fluent: Experiment with name 'verifier_guided_reasoning_demo' does not exist. Creating a new experiment.


{'num_results': 4,
 'status_counts': {'pass': 3, 'fail': 1},
 'error_counts': {'numeric_inconsistency': 1},
 'pass_rate': 0.75,
 'num_traces': 2,
 'final_accuracy': 1.0,
 'traces': [{'sample_id': 'demo-good-1',
   'source_dataset': 'openai/gsm8k',
   'final_status': 'pass',
   'num_failures': 0,
   'error_types': []},
  {'sample_id': 'demo-bad-1',
   'source_dataset': 'openai/gsm8k',
   'final_status': 'pass',
   'num_failures': 1,
   'error_types': ['numeric_inconsistency']}]}

In [29]:
from pathlib import Path

report_path = Path('artifacts/eval/demo_report.md')
print(report_path.read_text(encoding='utf-8'))


# Verifier-Guided Reasoning Demo Report

## Summary
- Traces evaluated: 2
- Final-answer accuracy: 100.00%
- Mean verifier pass rate: 75.00%

## Status Counts
- fail: 1
- pass: 3

## Error Counts
- numeric_inconsistency: 1

## Trace Diagnostics
- `demo-good-1` (openai/gsm8k): final=pass, failures=0, errors=[]
- `demo-bad-1` (openai/gsm8k): final=pass, failures=1, errors=['numeric_inconsistency']



## Arithmetic SFT Path

The public MVP stays arithmetic-first:

- `MU-NLPC/Calc-svamp` for seed structured traces
- `openai/gsm8k` for benchmark and augmentation
- verifier gates before SFT export
- `Qwen/Qwen2.5-1.5B-Instruct` as the default base model
- `Qwen/Qwen2.5-Math-1.5B-Instruct` as the optional comparison model

Do not move to DPO until the verifier and accepted-trace pipeline are stable.


In [30]:
TRAINING_CONFIG = {
    'base_model': 'Qwen/Qwen2.5-1.5B-Instruct',
    'comparison_model': 'Qwen/Qwen2.5-Math-1.5B-Instruct',
    'tuning_method': 'qlora',
    'max_seq_length': 2048,
    'best_of_n': 4,
    'datasets': {
        'seed': 'MU-NLPC/Calc-svamp',
        'benchmark': 'openai/gsm8k',
        'logic_eval': 'tasksource/folio',
    },
}
TRAINING_CONFIG


{'base_model': 'Qwen/Qwen2.5-1.5B-Instruct',
 'comparison_model': 'Qwen/Qwen2.5-Math-1.5B-Instruct',
 'tuning_method': 'qlora',
 'max_seq_length': 2048,
 'best_of_n': 4,
 'datasets': {'seed': 'MU-NLPC/Calc-svamp',
  'benchmark': 'openai/gsm8k',
  'logic_eval': 'tasksource/folio'}}

In [31]:
# Optional next step once you are ready for dataset-backed runs:
# !python scripts/prepare_datasets.py --demo --output data/processed/demo_arithmetic.jsonl
# !python scripts/run_small_demo.py --input data/processed/demo_arithmetic.jsonl \
#     --output artifacts/eval/demo_summary.json \
#     --report-md artifacts/eval/demo_report.md \
#     --tracker-root "$MLFLOW_TRACKING_URI"

print('Notebook scaffold complete. Move into real SFT only after accepted traces are stable.')


Notebook scaffold complete. Move into real SFT only after accepted traces are stable.
